Can chaning weights internally help ask to know is it En/Ett word in swedish


In [1]:
# Step 1
!pip -q install transformers accelerate

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

model.eval()



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [2]:
# Step 2

from sklearn.model_selection import train_test_split

# Download the Swedish KELLY dataset
url = "https://huggingface.co/datasets/codesue/kelly/resolve/main/sv.csv"
df = pd.read_csv(url)

# Keep nouns with a known en or ett article
nouns = df[df["POS"].isin(["noun-en", "noun-ett"])].copy()

nouns = nouns[["Lemma", "POS"]].dropna().drop_duplicates()

nouns["article"] = nouns["POS"].map({
    "noun-en": "en",
    "noun-ett": "ett"
})

nouns = nouns[["Lemma", "article"]]

# Remove nouns that have conflicting article labels
counts = nouns.groupby("Lemma")["article"].nunique()
ambiguous = counts[counts > 1].index
nouns = nouns[~nouns["Lemma"].isin(ambiguous)].copy()

# Split into training (70%), validation (15%), and test (15%)
train_df, remaining_df = train_test_split(
    nouns, test_size=0.30, random_state=42,
    stratify=nouns["article"]
)

val_df, test_df = train_test_split(
    remaining_df, test_size=0.50, random_state=42,
    stratify=remaining_df["article"]
)




In [3]:

# Step 3
def article_score(word, article):
    prompt = f"Vilken artikel har det svenska substantivet '{word}'? Svar:"

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    ).input_ids

    answer_ids = tokenizer(
        " " + article, add_special_tokens=False
    ).input_ids

    input_ids = torch.tensor(
        [prompt_ids + answer_ids],
        device=model.device
    )

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits.float()
        log_probs = torch.log_softmax(logits, dim=-1)

    score = 0.0
    for i in range(len(prompt_ids), input_ids.shape[1]):
        score += log_probs[0, i - 1, input_ids[0, i]].item()

    return score


# Choose 10 en nouns and 10 ett nouns from validation data
sample = pd.concat([
    val_df[val_df["article"] == "en"].sample(10, random_state=42),
    val_df[val_df["article"] == "ett"].sample(10, random_state=42)
])

results = []

for _, row in sample.iterrows():
    word = row["Lemma"]

    en_score = article_score(word, "en")
    ett_score = article_score(word, "ett")

    prediction = "ett" if ett_score > en_score else "en"

    results.append({
        "noun": word,
        "correct": row["article"],
        "predicted": prediction,
        "is_correct": prediction == row["article"]
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))
print("\nAccuracy:", f"{results_df['is_correct'].mean():.1%}")

for article in ["en", "ett"]:
    group = results_df[results_df["correct"] == article]
    print(f"{article} accuracy:", f"{group['is_correct'].mean():.1%}")


                           noun correct predicted  is_correct
                     försäkring      en       ett       False
                          trupp      en       ett       False
                            dos      en       ett       False
socialdemokrat (vardagl. sosse)      en        en        True
                levnadsstandard      en       ett       False
                         server      en       ett       False
                         kolumn      en       ett       False
                       filosofi      en       ett       False
                            gud      en       ett       False
                     sexualitet      en       ett       False
                           lamm     ett       ett        True
                     testamente     ett       ett        True
                     narkotikum     ett        en       False
                        utbrott     ett       ett        True
                           korn     ett       ett        True
        

In [4]:
# Step 3 continue

sample = pd.concat([
    val_df[val_df["article"] == "en"].sample(100, random_state=42),
    val_df[val_df["article"] == "ett"].sample(100, random_state=42)
])

# Calculate scores without changing model weights
results = []

for _, row in sample.iterrows():
    word = row["Lemma"]

    en_score = article_score(word, "en")
    ett_score = article_score(word, "ett")

    results.append({
        "correct": row["article"],
        "difference": ett_score - en_score
    })

scores_df = pd.DataFrame(results)

# Compare different thresholds
for threshold in [0, 0.5, 1, 1.5, 2]:
    predictions = scores_df["difference"].apply(
        lambda x: "ett" if x > threshold else "en"
    )

    en_mask = scores_df["correct"] == "en"
    ett_mask = scores_df["correct"] == "ett"

    en_acc = (predictions[en_mask] == "en").mean()
    ett_acc = (predictions[ett_mask] == "ett").mean()

    balanced_acc = (en_acc + ett_acc) / 2

    print(
        f"Threshold {threshold}: "
        f"en={en_acc:.0%}, "
        f"ett={ett_acc:.0%}, "
        f"balanced={balanced_acc:.0%}"
    )


Threshold 0: en=25%, ett=76%, balanced=50%
Threshold 0.5: en=73%, ett=37%, balanced=55%
Threshold 1: en=96%, ett=5%, balanced=50%
Threshold 1.5: en=99%, ett=1%, balanced=50%
Threshold 2: en=100%, ett=0%, balanced=50%


In [5]:
# Step 4

layer_number = 14
activations = []


train_sample = pd.concat([
    train_df[train_df["article"] == "en"].sample(10, random_state=42),
    train_df[train_df["article"] == "ett"].sample(10, random_state=42)
])


def save_activation(module, inputs):
    activations.append(inputs[0][0, -1].detach().float().cpu())

hook = model.model.layers[layer_number].mlp.down_proj.register_forward_pre_hook(
    save_activation
)

try:
    for _, row in train_sample.iterrows():
        word = row["Lemma"]
        prompt = f"Vilken artikel har det svenska substantivet '{word}'? Svar:"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            model(**inputs)
finally:
    hook.remove()


en_mean = torch.stack(activations[:10]).mean(dim=0)
ett_mean = torch.stack(activations[10:]).mean(dim=0)

differences = (en_mean - ett_mean).abs()

candidate_dimension = differences.argmax().item()

print("Layer:", layer_number)
print("Candidate dimension:", candidate_dimension)
print("Activation difference:", round(differences.max().item(), 4))


Layer: 14
Candidate dimension: 1463
Activation difference: 0.2687


In [6]:
# Step 5

layer = model.model.layers[14].mlp.down_proj
dimension = candidate_dimension
threshold = 0.5

original_weights = layer.weight[:, dimension].detach().clone()

def evaluate_model():
    correct_en = 0
    correct_ett = 0
    total_en = 0
    total_ett = 0

    for _, row in sample.iterrows():
        word = row["Lemma"]
        correct = row["article"]

        en_score = article_score(word, "en")
        ett_score = article_score(word, "ett")

        prediction = (
            "ett" if ett_score - en_score > threshold else "en"
        )

        if correct == "en":
            total_en += 1
            correct_en += prediction == correct
        else:
            total_ett += 1
            correct_ett += prediction == correct

    en_acc = correct_en / total_en
    ett_acc = correct_ett / total_ett

    return (en_acc + ett_acc) / 2

# Measure original Qwen
baseline = evaluate_model()

try:
    with torch.no_grad():
        layer.weight[:, dimension].mul_(0.95)

    edited = evaluate_model()

finally:
    with torch.no_grad():
        layer.weight[:, dimension].copy_(original_weights)

print(f"Original Qwen: {baseline:.1%}")
print(f"Edited Qwen:   {edited:.1%}")
print(f"Difference:    {edited - baseline:+.1%}")
print("Original weights restored.")


Original Qwen: 55.0%
Edited Qwen:   54.0%
Difference:    -1.0%
Original weights restored.


In [7]:
# Step 6:

layer = model.model.layers[14].mlp.down_proj

generator = torch.Generator().manual_seed(42)
all_dimensions = torch.randperm(
    layer.weight.shape[1], generator=generator
).tolist()

random_dimensions = [
    d for d in all_dimensions if d != candidate_dimension
][:3]

random_results = []

for dimension in random_dimensions:

    original = layer.weight[:, dimension].detach().clone()

    try:
        with torch.no_grad():
            layer.weight[:, dimension].mul_(0.95)

        accuracy = evaluate_model()

        random_results.append(accuracy)

    finally:
        with torch.no_grad():
            layer.weight[:, dimension].copy_(original)


for dimension, accuracy in zip(random_dimensions, random_results):
    print(
        f"Random dimension {dimension}:",
        f"{accuracy:.1%}"
    )

print(
    "Average random edit:",
    f"{sum(random_results) / len(random_results):.1%}"
)



Random dimension 102: 53.0%
Random dimension 1358: 55.5%
Random dimension 2678: 56.0%
Average random edit: 54.8%


In [8]:
# Step 7

layer = model.model.layers[14].mlp.down_proj
dimension = candidate_dimension
threshold = 0.5

def evaluate_test():
    predictions = []

    for _, row in test_df.iterrows():
        word = row["Lemma"]

        en_score = article_score(word, "en")
        ett_score = article_score(word, "ett")

        prediction = (
            "ett" if ett_score - en_score > threshold else "en"
        )

        predictions.append(prediction)

    correct = test_df["article"].reset_index(drop=True)
    predicted = pd.Series(predictions)

    en_accuracy = (predicted[correct == "en"] == "en").mean()
    ett_accuracy = (predicted[correct == "ett"] == "ett").mean()

    balanced_accuracy = (en_accuracy + ett_accuracy) / 2
    overall_accuracy = (predicted == correct).mean()

    return overall_accuracy, balanced_accuracy, en_accuracy, ett_accuracy


# Evaluate original Qwen
original_result = evaluate_test()

saved_weights = layer.weight[:, dimension].detach().clone()

try:
    with torch.no_grad():
        layer.weight[:, dimension].mul_(0.95)

    edited_result = evaluate_test()

finally:
    with torch.no_grad():
        layer.weight[:, dimension].copy_(saved_weights)


always_en_accuracy = (test_df["article"] == "en").mean()

print("FINAL TEST RESULTS")
print("------------------")

for name, result in [
    ("Original Qwen", original_result),
    ("Targeted edit", edited_result)
]:
    overall, balanced, en_acc, ett_acc = result

    print(f"\n{name}")
    print(f"Overall accuracy:  {overall:.1%}")
    print(f"Balanced accuracy: {balanced:.1%}")
    print(f"en accuracy:       {en_acc:.1%}")
    print(f"ett accuracy:      {ett_acc:.1%}")

print("\nAlways-en baseline")
print(f"Overall accuracy:  {always_en_accuracy:.1%}")
print("Balanced accuracy: 50.0%")

print("\nOriginal weights restored.")


FINAL TEST RESULTS
------------------

Original Qwen
Overall accuracy:  64.5%
Balanced accuracy: 54.6%
en accuracy:       73.2%
ett accuracy:      36.1%

Targeted edit
Overall accuracy:  64.8%
Balanced accuracy: 55.0%
en accuracy:       73.4%
ett accuracy:      36.7%

Always-en baseline
Overall accuracy:  76.6%
Balanced accuracy: 50.0%

Original weights restored.


In [10]:
# Step 8

import transformers
from huggingface_hub import model_info

revision = model_info(model_name).sha

print("Model:", model_name)
print("Model revision:", revision)
print("Dataset split seed:", 42)
print("Random control seed:", 42)
print("Edited layer:", 14)
print("Edited dimension:", candidate_dimension)
print("Weight multiplier:", 0.95)
print("Decision threshold:", 0.5)
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)


Model: Qwen/Qwen3-0.6B
Model revision: c1899de289a04d12100db370d81485cdf75e47ca
Dataset split seed: 42
Random control seed: 42
Edited layer: 14
Edited dimension: 1463
Weight multiplier: 0.95
Decision threshold: 0.5
PyTorch version: 2.14.1+cu130
Transformers version: 5.22.1
